### #9

Kaggle competition: [\[link\]](https://www.kaggle.com/competitions/playground-series-s5e9/)

Entry by Robin R.P.M. Kras

robkras.com

### ⭐ 1. Introduction & Overview


Your Goal: The goal of this competition is to predict a song's beats-per-minute.

### 🔹 2. Import Libraries & Set Up


In [37]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-

# =====================
# General utilities
# =====================
import json
import os
import pickle
import time
from collections import Counter

# =====================
# Data handling & processing
# =====================
import numpy as np
import pandas as pd
from tqdm import tqdm

# =====================
# Visualization
# =====================
import matplotlib.pyplot as plt
import seaborn as sns

# =====================
# Machine Learning - Core scikit-learn
# =====================
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, ElasticNet
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score,
    root_mean_squared_error, roc_auc_score
)
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.svm import SVC, SVR

# =====================
# Machine Learning - Tree Boosting & advanced
# =====================
import xgboost as xg
import lightgbm as lgb
import catboost

# =====================
# Deep Learning - TensorFlow / Keras
# =====================
import tensorflow as tf
from keras import regularizers
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.optimizers import Adam

# =====================
# Deep Learning - PyTorch
# =====================
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torch.optim as optim

# =====================
# Imbalanced data handling
# =====================
from imblearn.over_sampling import SMOTE

# =====================
# Optimization / AutoML
# =====================
import optuna

# =====================
# Feature importance & explainability
# =====================
import shap

# =====================
# Self Made Utilities
# =====================
from utils import *

# =====================
# Settings & reproducibility
# =====================
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

print("Libraries successfully loaded. Ready to go!")

Libraries successfully loaded. Ready to go!


In [38]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [39]:
train.head()

,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0,0.603610,-7.636942,0.023500,0.000005,0.000001,0.051385,0.409866,290715.6450,0.826267,147.53020
1,1,0.639451,-16.267598,0.071520,0.444929,0.349414,0.170522,0.651010,164519.5174,0.145400,136.15963
2,2,0.514538,-15.953575,0.110715,0.173699,0.453814,0.029576,0.423865,174495.5667,0.624667,55.31989
3,3,0.734463,-1.357000,0.052965,0.001651,0.159717,0.086366,0.278745,225567.4651,0.487467,147.91212
4,4,0.532968,-13.056437,0.023500,0.068687,0.000001,0.331345,0.477769,213960.6789,0.947333,89.58511


In [ ]:
def create_features(df):
    df_new = df.copy()
    
    df_new['Rhythm_Energy'] = df_new['RhythmScore'] * df_new['Energy']
    df_new['Rhythm_Loudness'] = df_new['RhythmScore'] * df_new['AudioLoudness']
    
    df_new['Duration_Minutes'] = df_new['TrackDurationMs'] / 60000  # Convert to minutes
    df_new['Duration_Energy_Ratio'] = df_new['TrackDurationMs'] / (df_new['Energy'] * 10000 + 1) 
    
    df_new['RhythmScore_Squared'] = df_new['RhythmScore'] ** 2
    df_new['Energy_Squared'] = df_new['Energy'] ** 2
    df_new['Log_Duration'] = np.log1p(df_new['TrackDurationMs'])  # log(1+x) to handle zeros
    
    df_new['Acoustic_Instrumental_Ratio'] = df_new['AcousticQuality'] / (df_new['InstrumentalScore'] + 0.01) 
    df_new['Vocal_Energy'] = df_new['VocalContent'] * df_new['Energy']
    
    df_new['Live_Energy'] = df_new['LivePerformanceLikelihood'] * df_new['Energy']
    df_new['Mood_Rhythm'] = df_new['MoodScore'] * df_new['RhythmScore']
    
    df_new['Audio_Intensity'] = (df_new['Energy'] * np.abs(df_new['AudioLoudness'])) / 10 
    df_new['Performance_Character'] = (df_new['LivePerformanceLikelihood'] + df_new['MoodScore']) / 2
    
    df_new['Energy_Loudness_Ratio'] = df_new['Energy'] / (np.abs(df_new['AudioLoudness']) + 0.01)
    df_new['Rhythm_Duration_Density'] = df_new['RhythmScore'] / df_new['Duration_Minutes']

    # new below

    df_new['Log_RhythmScore'] = np.log1p(df_new['RhythmScore'])
    df_new['Log_Energy'] = np.log1p(df_new['Energy'])
    df_new['Log_AcousticQuality'] = np.log1p(df_new['AcousticQuality'])
    df_new['Log_InstrumentalScore'] = np.log1p(df_new['InstrumentalScore'])
    df_new['Log_VocalContent'] = np.log1p(df_new['VocalContent'])
    df_new['Log_LivePerformanceLikelihood'] = np.log1p(df_new['LivePerformanceLikelihood'])
    df_new['Log_MoodScore'] = np.log1p(df_new['MoodScore'])
    df_new['Log_AudioLoudness'] = np.log1p(np.abs(df_new['AudioLoudness']) + 1)
    
    return df_new

train = create_features(train)
test = create_features(test)

In [41]:
X = train.drop(["id", "BeatsPerMinute"], axis=1)
y = train["BeatsPerMinute"]

X_test = test.drop(["id"], axis=1)

In [42]:
for col in train.columns:
    if train[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        X_test[col] = le.transform(X_test[col])

In [43]:
import lightgbm as lgb
from sklearn.model_selection import KFold

# Use 10-fold stratified cross-validation
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
y_probs_lgbm = np.zeros(len(X_test))
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    lightgbm = lgb.LGBMRegressor(
        n_estimators=20000,
        learning_rate=0.06,
        num_leaves=100,
        max_depth=10,
        min_child_samples=9,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.78,
        reg_lambda=3.0,
        random_state=42,
        verbosity=-1,
        device="gpu",
        gpu_platform_id=0,
        gpu_device_id=0
    )
    
    lightgbm.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(period=500)
        ]
    )

    models.append(lightgbm)
    
    # Average predictions across all folds
    y_probs_lgbm += lightgbm.predict(X_test) / n_splits

best_rmse = root_mean_squared_error(y, lightgbm.predict(X))
print(f"\nBest RMSE: {best_rmse:.4f}")

Training fold 1/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[23]	valid_0's l2: 698.886
Training fold 2/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[21]	valid_0's l2: 699.214
Training fold 3/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[27]	valid_0's l2: 703.021
Training fold 4/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[22]	valid_0's l2: 699.95
Training fold 5/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[35]	valid_0's l2: 703.343
Training fold 6/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[15]	valid_0's l2: 703.641
Training fold 7/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:

In [44]:
import xgboost as xgb
from sklearn.model_selection import KFold

# Use 10-fold stratified cross-validation
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
y_probs_xgb = np.zeros(len(X_test))
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    xgboost = xgb.XGBRegressor(
        n_estimators=20000,
        learning_rate=0.06,
        max_depth=10,
        min_child_weight=9,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.78,
        reg_lambda=3.0,
        random_state=42,
        early_stopping_rounds=100,
        verbosity=1,
        tree_method='gpu_hist',  # Use GPU acceleration
        gpu_id=0
    )
    
    xgboost.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
    )

    models.append(xgboost)
    
    # Average predictions across all folds
    y_probs_xgb += xgboost.predict(X_test) / n_splits
    
best_rmse = root_mean_squared_error(y, xgboost.predict(X))
print(f"\nBest RMSE: {best_rmse:.4f}")

Training fold 1/10 >>>
[0]	validation_0-rmse:26.44166
[1]	validation_0-rmse:26.44167
[2]	validation_0-rmse:26.44131
[3]	validation_0-rmse:26.44147
[4]	validation_0-rmse:26.44224
[5]	validation_0-rmse:26.44249
[6]	validation_0-rmse:26.44291
[7]	validation_0-rmse:26.44329
[8]	validation_0-rmse:26.44303
[9]	validation_0-rmse:26.44140
[10]	validation_0-rmse:26.44264
[11]	validation_0-rmse:26.44296
[12]	validation_0-rmse:26.44258
[13]	validation_0-rmse:26.44246
[14]	validation_0-rmse:26.44200
[15]	validation_0-rmse:26.44282
[16]	validation_0-rmse:26.44407
[17]	validation_0-rmse:26.44279
[18]	validation_0-rmse:26.44316
[19]	validation_0-rmse:26.44479
[20]	validation_0-rmse:26.44477
[21]	validation_0-rmse:26.44529
[22]	validation_0-rmse:26.44550
[23]	validation_0-rmse:26.44599
[24]	validation_0-rmse:26.44670
[25]	validation_0-rmse:26.44755
[26]	validation_0-rmse:26.44844
[27]	validation_0-rmse:26.44804
[28]	validation_0-rmse:26.44813
[29]	validation_0-rmse:26.44765
[30]	validation_0-rmse:26.4

Best RMSE: 26.4178


In [45]:
import catboost as cb
from sklearn.model_selection import KFold

# Use 10-fold stratified cross-validation
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
y_probs_cb = np.zeros(len(X_test))
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    catboost = cb.CatBoostRegressor(
        iterations=15000,
        learning_rate=0.08,
        depth=8,
        min_data_in_leaf=20,
        subsample=0.85,
        #colsample_bylevel=0.8,
        random_strength=1.5,
        reg_lambda=1.5,
        bootstrap_type='Bernoulli',
        random_seed=42,
        early_stopping_rounds=100,
        verbose=500,
        devices='0',
        task_type='GPU',  # Use GPU for training
    )
    
    catboost.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
    )

    models.append(catboost)
    
    # Average predictions across all folds
    y_probs_cb += catboost.predict(X_test) / n_splits

best_rmse = root_mean_squared_error(y, catboost.predict(X))
print(f"\nBest RMSE: {best_rmse:.4f}")

Training fold 1/10 >>>
0:	learn: 26.4682245	test: 26.4412176	best: 26.4412176 (0)	total: 96.8ms	remaining: 24m 11s
bestTest = 26.43360597
bestIteration = 46
Shrink model to first 47 iterations.
Training fold 2/10 >>>
0:	learn: 26.4677747	test: 26.4470339	best: 26.4470339 (0)	total: 7.7ms	remaining: 1m 55s
bestTest = 26.44257837
bestIteration = 11
Shrink model to first 12 iterations.
Training fold 3/10 >>>
0:	learn: 26.4593133	test: 26.5213678	best: 26.5213678 (0)	total: 9.23ms	remaining: 2m 18s
bestTest = 26.51266525
bestIteration = 28
Shrink model to first 29 iterations.
Training fold 4/10 >>>
0:	learn: 26.4657115	test: 26.4627591	best: 26.4627591 (0)	total: 7.31ms	remaining: 1m 49s
bestTest = 26.4572117
bestIteration = 22
Shrink model to first 23 iterations.
Training fold 5/10 >>>
0:	learn: 26.4583803	test: 26.5291915	best: 26.5291915 (0)	total: 7.67ms	remaining: 1m 55s
bestTest = 26.52163514
bestIteration = 25
Shrink model to first 26 iterations.
Training fold 6/10 >>>
0:	learn: 26.

In [10]:
output_lgbm = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': y_probs_lgbm
})

output_lgbm.to_csv('attempt-lightgbm1.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


In [11]:
output_lgbm.head()

,id,BeatsPerMinute
0,524164,119.014855
1,524165,118.575555
2,524166,119.342685
3,524167,119.113536
4,524168,119.407333


Stacking ensemble

In [46]:
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_lgbm = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

y_probs_lgbm = np.zeros(len(X_test))
y_probs_xgb = np.zeros(len(X_test))
y_probs_cb = np.zeros(len(X_test))

lgbm_models = []
xgb_models = []
cb_models = []

print("="*50)
print("TRAINING LIGHTGBM")
print("="*50)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    lightgbm = lgb.LGBMRegressor(
        n_estimators=20000,
        learning_rate=0.06,
        num_leaves=100,
        max_depth=10,
        min_child_samples=9,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.78,
        reg_lambda=3.0,
        random_state=42,
        verbosity=-1,
        device="gpu",
        gpu_platform_id=0,
        gpu_device_id=0
    )
    
    lightgbm.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(period=0)  # Silent
        ]
    )

    lgbm_models.append(lightgbm)
    
    oof_lgbm[val_idx] = lightgbm.predict(X_val)
    
    y_probs_lgbm += lightgbm.predict(X_test) / n_splits

lgbm_rmse = np.sqrt(mean_squared_error(y, oof_lgbm))
print(f"LightGBM OOF RMSE: {lgbm_rmse:.4f}")

print("="*50)
print("TRAINING XGBOOST")
print("="*50)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    xgboost = xgb.XGBRegressor(
        n_estimators=20000,
        learning_rate=0.06,
        max_depth=10,
        min_child_weight=9,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.78,
        reg_lambda=3.0,
        random_state=42,
        early_stopping_rounds=100,
        verbosity=0, 
        tree_method='gpu_hist',
        gpu_id=0
    )
    
    xgboost.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
        verbose=False
    )

    xgb_models.append(xgboost)
    
    oof_xgb[val_idx] = xgboost.predict(X_val)
    
    y_probs_xgb += xgboost.predict(X_test) / n_splits

xgb_rmse = np.sqrt(mean_squared_error(y, oof_xgb))
print(f"XGBoost OOF RMSE: {xgb_rmse:.4f}")

print("="*50)
print("TRAINING CATBOOST")
print("="*50)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    catboost = cb.CatBoostRegressor(
        iterations=15000,
        learning_rate=0.08,
        depth=8,
        min_data_in_leaf=20,
        subsample=0.85,
        random_strength=1.5,
        reg_lambda=1.5,
        bootstrap_type='Bernoulli',
        random_seed=42,
        early_stopping_rounds=100,
        verbose=0,  # Silent
        devices='0',
        task_type='GPU',
    )
    
    catboost.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
    )

    cb_models.append(catboost)
    
    oof_cb[val_idx] = catboost.predict(X_val)
    
    y_probs_cb += catboost.predict(X_test) / n_splits

cb_rmse = np.sqrt(mean_squared_error(y, oof_cb))
print(f"CatBoost OOF RMSE: {cb_rmse:.4f}")

print("="*50)
print("CREATING STACKED ENSEMBLE")
print("="*50)

stack_train = np.column_stack([oof_lgbm, oof_xgb, oof_cb])
stack_test = np.column_stack([y_probs_lgbm, y_probs_xgb, y_probs_cb])

print(f"Stacking features shape: {stack_train.shape}")
print(f"Individual model correlations:")
print(f"LGBM vs XGB: {np.corrcoef(oof_lgbm, oof_xgb)[0,1]:.4f}")
print(f"LGBM vs CB: {np.corrcoef(oof_lgbm, oof_cb)[0,1]:.4f}")
print(f"XGB vs CB: {np.corrcoef(oof_xgb, oof_cb)[0,1]:.4f}")

meta_models = {
    'Ridge': Ridge(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'LightGBM_Meta': lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, verbosity=-1)
}

kf_meta = KFold(n_splits=5, shuffle=True, random_state=42)
meta_results = {}

for meta_name, meta_model in meta_models.items():
    print(f"\nTesting {meta_name} as meta-model...")
    meta_oof = np.zeros(len(stack_train))
    meta_test_preds = np.zeros(len(stack_test))
    
    for fold, (train_idx, val_idx) in enumerate(kf_meta.split(stack_train)):
        X_meta_train = stack_train[train_idx]
        y_meta_train = y.iloc[train_idx]
        X_meta_val = stack_train[val_idx]
        
        meta_model_fold = meta_model.__class__(**meta_model.get_params())
        meta_model_fold.fit(X_meta_train, y_meta_train)
        
        meta_oof[val_idx] = meta_model_fold.predict(X_meta_val)
        meta_test_preds += meta_model_fold.predict(stack_test) / 5
    
    meta_rmse = np.sqrt(mean_squared_error(y, meta_oof))
    meta_results[meta_name] = {
        'rmse': meta_rmse,
        'oof': meta_oof,
        'test_preds': meta_test_preds
    }
    print(f"{meta_name} Meta-model RMSE: {meta_rmse:.4f}")

simple_avg_oof = (oof_lgbm + oof_xgb + oof_cb) / 3
simple_avg_test = (y_probs_lgbm + y_probs_xgb + y_probs_cb) / 3
simple_avg_rmse = np.sqrt(mean_squared_error(y, simple_avg_oof))

rmses = np.array([lgbm_rmse, xgb_rmse, cb_rmse])
weights = 1 / rmses
weights = weights / weights.sum()

weighted_avg_oof = (weights[0] * oof_lgbm + weights[1] * oof_xgb + weights[2] * oof_cb)
weighted_avg_test = (weights[0] * y_probs_lgbm + weights[1] * y_probs_xgb + weights[2] * y_probs_cb)
weighted_avg_rmse = np.sqrt(mean_squared_error(y, weighted_avg_oof))

print("="*50)
print("ENSEMBLE RESULTS SUMMARY")
print("="*50)
print(f"LightGBM RMSE:     {lgbm_rmse:.4f}")
print(f"XGBoost RMSE:      {xgb_rmse:.4f}")
print(f"CatBoost RMSE:     {cb_rmse:.4f}")
print(f"Simple Average:    {simple_avg_rmse:.4f}")
print(f"Weighted Average:  {weighted_avg_rmse:.4f}")

for meta_name, results in meta_results.items():
    print(f"{meta_name:15s}: {results['rmse']:.4f}")

best_meta = min(meta_results.items(), key=lambda x: x[1]['rmse'])
best_meta_name, best_meta_results = best_meta

print(f"\nBest ensemble: {best_meta_name} with RMSE: {best_meta_results['rmse']:.4f}")

final_predictions = best_meta_results['test_preds']

if best_meta_name == 'Ridge':
    ridge_meta = Ridge(alpha=1.0)
    ridge_meta.fit(stack_train, y)
    feature_importance = ridge_meta.coef_
    print(f"\nMeta-model feature weights:")
    print(f"LightGBM: {feature_importance[0]:.4f}")
    print(f"XGBoost:  {feature_importance[1]:.4f}")
    print(f"CatBoost: {feature_importance[2]:.4f}")

print("\n" + "="*50)
print("ADVANCED: MULTI-LEVEL STACKING")
print("="*50)

level2_train = np.column_stack([
    stack_train,
    best_meta_results['oof'].reshape(-1, 1),  
])

level2_test = np.column_stack([
    stack_test,
    best_meta_results['test_preds'].reshape(-1, 1)
])

# Level 2 meta-model
level2_meta = Ridge(alpha=0.5)
level2_oof = np.zeros(len(level2_train))
level2_test_preds = np.zeros(len(level2_test))

kf_level2 = KFold(n_splits=3, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf_level2.split(level2_train)):
    X_l2_train = level2_train[train_idx]
    y_l2_train = y.iloc[train_idx]
    X_l2_val = level2_train[val_idx]
    
    level2_meta_fold = Ridge(alpha=0.5)
    level2_meta_fold.fit(X_l2_train, y_l2_train)
    
    level2_oof[val_idx] = level2_meta_fold.predict(X_l2_val)
    level2_test_preds += level2_meta_fold.predict(level2_test) / 3

level2_rmse = np.sqrt(mean_squared_error(y, level2_oof))
print(f"Level 2 Stacking RMSE: {level2_rmse:.4f}")

# Choose final model
if level2_rmse < best_meta_results['rmse']:
    print("Level 2 stacking is better!")
    final_predictions = level2_test_preds
else:
    print(f"Level 1 {best_meta_name} is better!")
    final_predictions = best_meta_results['test_preds']

print(f"\nFinal ensemble RMSE: {min(level2_rmse, best_meta_results['rmse']):.4f}")

TRAINING LIGHTGBM
Training fold 1/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[27]	valid_0's l2: 698.885
Training fold 2/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[21]	valid_0's l2: 699.214
Training fold 3/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[27]	valid_0's l2: 703.021
Training fold 4/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[22]	valid_0's l2: 699.952
Training fold 5/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[35]	valid_0's l2: 703.343
Training fold 6/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[18]	valid_0's l2: 703.641
Training fold 7/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping,

Level 2 Stacking RMSE: 26.4583


In [13]:
print("="*50)
print("SAVING PREDICTIONS")
print("="*50)

# Save individual model predictions
output_lgbm = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': y_probs_lgbm
})

output_xgb = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': y_probs_xgb
})

output_cb = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': y_probs_cb
})

# Save ensemble predictions
output_simple_avg = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': simple_avg_test
})

output_weighted_avg = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': weighted_avg_test
})

output_best_meta = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': best_meta_results['test_preds']
})

# Final ensemble prediction (Level 2 if better, otherwise best Level 1)
if level2_rmse < best_meta_results['rmse']:
    final_predictions = level2_test_preds
    ensemble_type = "Level2_Stacking"
else:
    final_predictions = best_meta_results['test_preds']
    ensemble_type = f"Level1_{best_meta_name}"

output_final = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': final_predictions
})

# Save all predictions
output_lgbm.to_csv('attempt-lightgbm.csv', index=False)
print("LightGBM submission saved!")

output_xgb.to_csv('attempt-xgboost.csv', index=False)
print("XGBoost submission saved!")

output_cb.to_csv('attempt-catboost.csv', index=False)
print("CatBoost submission saved!")

output_simple_avg.to_csv('attempt-simple-average.csv', index=False)
print("Simple Average submission saved!")

output_weighted_avg.to_csv('attempt-weighted-average.csv', index=False)
print("Weighted Average submission saved!")

output_best_meta.to_csv(f'attempt-{best_meta_name.lower()}-meta.csv', index=False)
print(f"{best_meta_name} Meta-model submission saved!")

output_final.to_csv(f'attempt-final-{ensemble_type.lower()}.csv', index=False)
print(f"Final {ensemble_type} submission saved!")

# Also save a summary of all ensemble methods for comparison
ensemble_summary = pd.DataFrame({
    'id': test.id,
    'LightGBM': y_probs_lgbm,
    'XGBoost': y_probs_xgb,
    'CatBoost': y_probs_cb,
    'Simple_Average': simple_avg_test,
    'Weighted_Average': weighted_avg_test,
    'Best_Meta': best_meta_results['test_preds'],
    'Final_Ensemble': final_predictions
})

ensemble_summary.to_csv('all-predictions-comparison.csv', index=False)
print("All predictions comparison saved!")

print("\n" + "="*50)
print("SUBMISSION FILES CREATED:")
print("="*50)
print("Individual Models:")
print("- attempt-lightgbm.csv")
print("- attempt-xgboost.csv") 
print("- attempt-catboost.csv")
print("\nEnsemble Methods:")
print("- attempt-simple-average.csv")
print("- attempt-weighted-average.csv")
print(f"- attempt-{best_meta_name.lower()}-meta.csv")
print(f"- attempt-final-{ensemble_type.lower()}.csv")
print("\nComparison:")
print("- all-predictions-comparison.csv")

print(f"\nRecommended submission: attempt-final-{ensemble_type.lower()}.csv")
print(f"Expected performance: {min(level2_rmse, best_meta_results['rmse']):.4f} RMSE")

SAVING PREDICTIONS
LightGBM submission saved!
XGBoost submission saved!
CatBoost submission saved!
Simple Average submission saved!
Weighted Average submission saved!
Ridge Meta-model submission saved!
Final Level2_Stacking submission saved!
All predictions comparison saved!

SUBMISSION FILES CREATED:
Individual Models:
- attempt-lightgbm.csv
- attempt-xgboost.csv
- attempt-catboost.csv

Ensemble Methods:
- attempt-simple-average.csv
- attempt-weighted-average.csv
- attempt-ridge-meta.csv
- attempt-final-level2_stacking.csv

Comparison:
- all-predictions-comparison.csv

Recommended submission: attempt-final-level2_stacking.csv
Expected performance: 26.4582 RMSE


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import optuna
from functools import partial
import warnings
warnings.filterwarnings('ignore')

# Hyperparameter tuning functions
def objective_lgbm(trial, X, y, cv_folds=5):
    """Objective function for LightGBM hyperparameter tuning"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'verbosity': -1,
        'device': "gpu",
        'gpu_platform_id': 0,
        'gpu_device_id': 0
    }
    
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )
        
        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores)

def objective_xgb(trial, X, y, cv_folds=5):
    """Objective function for XGBoost hyperparameter tuning"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'early_stopping_rounds': 50,
        'verbosity': 0,
        'tree_method': 'gpu_hist',
        'gpu_id': 0
    }
    
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores)

def objective_cb(trial, X, y, cv_folds=5):
    """Objective function for CatBoost hyperparameter tuning"""
    params = {
        'iterations': trial.suggest_int('iterations', 1000, 4000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 4, 10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.5, 3.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'bootstrap_type': 'Bernoulli',
        'random_seed': 42,
        'early_stopping_rounds': 50,
        'verbose': 0,
        'devices': '0',
        'task_type': 'GPU',
    }
    
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = cb.CatBoostRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )
        
        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores)

def objective_meta(trial, stack_train, y, model_type):
    """Objective function for meta-model hyperparameter tuning"""
    if model_type == 'Ridge':
        params = {
            'alpha': trial.suggest_float('alpha', 0.01, 10.0, log=True)
        }
        model = Ridge(**params)
    elif model_type == 'ElasticNet':
        params = {
            'alpha': trial.suggest_float('alpha', 0.01, 10.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 0.9)
        }
        model = ElasticNet(**params)
    elif model_type == 'LightGBM_Meta':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'max_depth': trial.suggest_int('max_depth', 2, 8),
            'num_leaves': trial.suggest_int('num_leaves', 10, 100),
            'random_state': 42,
            'verbosity': -1
        }
        model = lgb.LGBMRegressor(**params)
    
    # Cross-validation for meta-model
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(stack_train):
        X_train, X_val = stack_train[train_idx], stack_train[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model_fold = model.__class__(**model.get_params())
        model_fold.fit(X_train, y_train)
        pred = model_fold.predict(X_val)
        
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores)

# Main training pipeline with hyperparameter tuning
def train_ensemble_with_tuning(X, y, X_test, tune_params=True, n_trials=100):
    """
    Complete ensemble training pipeline with hyperparameter tuning
    
    Args:
        X: Training features
        y: Training target
        X_test: Test features  
        tune_params: Whether to perform hyperparameter tuning
        n_trials: Number of optimization trials for each model
    """
    
    n_splits = 10
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Initialize arrays
    oof_lgbm = np.zeros(len(X))
    oof_xgb = np.zeros(len(X))
    oof_cb = np.zeros(len(X))
    
    y_probs_lgbm = np.zeros(len(X_test))
    y_probs_xgb = np.zeros(len(X_test))
    y_probs_cb = np.zeros(len(X_test))
    
    lgbm_models = []
    xgb_models = []
    cb_models = []
    
    # Step 1: Hyperparameter tuning (if enabled)
    if tune_params:
        print("="*60)
        print("HYPERPARAMETER TUNING PHASE")
        print("="*60)
        
        # Tune LightGBM
        print("Tuning LightGBM hyperparameters...")
        study_lgbm = optuna.create_study(direction='minimize', 
                                        sampler=optuna.samplers.TPESampler(seed=42))
        study_lgbm.optimize(partial(objective_lgbm, X=X, y=y), n_trials=n_trials, show_progress_bar=True)
        best_lgbm_params = study_lgbm.best_params
        print(f"Best LightGBM RMSE: {study_lgbm.best_value:.4f}")
        print(f"Best LightGBM params: {best_lgbm_params}")
        
        # Tune XGBoost
        print("\nTuning XGBoost hyperparameters...")
        study_xgb = optuna.create_study(direction='minimize',
                                       sampler=optuna.samplers.TPESampler(seed=42))
        study_xgb.optimize(partial(objective_xgb, X=X, y=y), n_trials=n_trials, show_progress_bar=True)
        best_xgb_params = study_xgb.best_params
        print(f"Best XGBoost RMSE: {study_xgb.best_value:.4f}")
        print(f"Best XGBoost params: {best_xgb_params}")
        
        # Tune CatBoost
        print("\nTuning CatBoost hyperparameters...")
        study_cb = optuna.create_study(direction='minimize',
                                      sampler=optuna.samplers.TPESampler(seed=42))
        study_cb.optimize(partial(objective_cb, X=X, y=y), n_trials=n_trials, show_progress_bar=True)
        best_cb_params = study_cb.best_params
        print(f"Best CatBoost RMSE: {study_cb.best_value:.4f}")
        print(f"Best CatBoost params: {best_cb_params}")
        
    else:
        # Use default hyperparameters from original code
        best_lgbm_params = {
            'n_estimators': 20000,
            'learning_rate': 0.06,
            'num_leaves': 100,
            'max_depth': 10,
            'min_child_samples': 9,
            'subsample': 0.8,
            'colsample_bytree': 0.5,
            'reg_alpha': 0.78,
            'reg_lambda': 3.0,
            'random_state': 42,
            'verbosity': -1,
            'device': "gpu",
            'gpu_platform_id': 0,
            'gpu_device_id': 0
        }
        
        best_xgb_params = {
            'n_estimators': 20000,
            'learning_rate': 0.06,
            'max_depth': 10,
            'min_child_weight': 9,
            'subsample': 0.8,
            'colsample_bytree': 0.5,
            'reg_alpha': 0.78,
            'reg_lambda': 3.0,
            'random_state': 42,
            'early_stopping_rounds': 100,
            'verbosity': 0,
            'tree_method': 'gpu_hist',
            'gpu_id': 0
        }
        
        best_cb_params = {
            'iterations': 15000,
            'learning_rate': 0.08,
            'depth': 8,
            'min_data_in_leaf': 20,
            'subsample': 0.85,
            'random_strength': 1.5,
            'reg_lambda': 1.5,
            'bootstrap_type': 'Bernoulli',
            'random_seed': 42,
            'early_stopping_rounds': 100,
            'verbose': 0,
            'devices': '0',
            'task_type': 'GPU',
        }
    
    # Step 2: Train base models with optimized hyperparameters
    print("\n" + "="*50)
    print("TRAINING LIGHTGBM WITH OPTIMIZED PARAMS")
    print("="*50)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"Training fold {fold + 1}/{n_splits} >>>")
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # Add GPU settings back if tuning removed them
        lgbm_params = best_lgbm_params.copy()
        lgbm_params.update({
            'random_state': 42,
            'verbosity': -1,
            'device': "gpu",
            'gpu_platform_id': 0,
            'gpu_device_id': 0
        })
        
        lightgbm = lgb.LGBMRegressor(**lgbm_params)
        
        lightgbm.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100),
                lgb.log_evaluation(period=0)
            ]
        )
        
        lgbm_models.append(lightgbm)
        oof_lgbm[val_idx] = lightgbm.predict(X_val)
        y_probs_lgbm += lightgbm.predict(X_test) / n_splits
    
    lgbm_rmse = np.sqrt(mean_squared_error(y, oof_lgbm))
    print(f"LightGBM OOF RMSE: {lgbm_rmse:.4f}")
    
    print("="*50)
    print("TRAINING XGBOOST WITH OPTIMIZED PARAMS")
    print("="*50)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"Training fold {fold + 1}/{n_splits} >>>")
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # Add GPU and other settings back if tuning removed them
        xgb_params = best_xgb_params.copy()
        xgb_params.update({
            'random_state': 42,
            'early_stopping_rounds': 100,
            'verbosity': 0,
            'tree_method': 'gpu_hist',
            'gpu_id': 0
        })
        
        xgboost = xgb.XGBRegressor(**xgb_params)
        
        xgboost.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        xgb_models.append(xgboost)
        oof_xgb[val_idx] = xgboost.predict(X_val)
        y_probs_xgb += xgboost.predict(X_test) / n_splits
    
    xgb_rmse = np.sqrt(mean_squared_error(y, oof_xgb))
    print(f"XGBoost OOF RMSE: {xgb_rmse:.4f}")
    
    print("="*50)
    print("TRAINING CATBOOST WITH OPTIMIZED PARAMS")
    print("="*50)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"Training fold {fold + 1}/{n_splits} >>>")
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # Add GPU and other settings back if tuning removed them
        cb_params = best_cb_params.copy()
        cb_params.update({
            'bootstrap_type': 'Bernoulli',
            'random_seed': 42,
            'early_stopping_rounds': 100,
            'verbose': 0,
            'devices': '0',
            'task_type': 'GPU',
        })
        
        catboost = cb.CatBoostRegressor(**cb_params)
        
        catboost.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )
        
        cb_models.append(catboost)
        oof_cb[val_idx] = catboost.predict(X_val)
        y_probs_cb += catboost.predict(X_test) / n_splits
    
    cb_rmse = np.sqrt(mean_squared_error(y, oof_cb))
    print(f"CatBoost OOF RMSE: {cb_rmse:.4f}")
    
    # Step 3: Create stacked ensemble with tuned meta-models
    print("="*50)
    print("CREATING STACKED ENSEMBLE WITH TUNED META-MODELS")
    print("="*50)
    
    stack_train = np.column_stack([oof_lgbm, oof_xgb, oof_cb])
    stack_test = np.column_stack([y_probs_lgbm, y_probs_xgb, y_probs_cb])
    
    print(f"Stacking features shape: {stack_train.shape}")
    print(f"Individual model correlations:")
    print(f"LGBM vs XGB: {np.corrcoef(oof_lgbm, oof_xgb)[0,1]:.4f}")
    print(f"LGBM vs CB: {np.corrcoef(oof_lgbm, oof_cb)[0,1]:.4f}")
    print(f"XGB vs CB: {np.corrcoef(oof_xgb, oof_cb)[0,1]:.4f}")
    
    meta_models_to_tune = ['Ridge', 'ElasticNet', 'LightGBM_Meta']
    meta_results = {}
    
    # Tune each meta-model
    if tune_params:
        print("\nTuning meta-models...")
        for meta_name in meta_models_to_tune:
            print(f"Tuning {meta_name}...")
            study_meta = optuna.create_study(direction='minimize',
                                           sampler=optuna.samplers.TPESampler(seed=42))
            study_meta.optimize(
                partial(objective_meta, stack_train=stack_train, y=y, model_type=meta_name), 
                n_trials=50, 
                show_progress_bar=True
            )
            
            best_meta_params = study_meta.best_params
            print(f"Best {meta_name} RMSE: {study_meta.best_value:.4f}")
            
            # Train final meta-model with best params
            if meta_name == 'Ridge':
                meta_model = Ridge(**best_meta_params)
            elif meta_name == 'ElasticNet':
                meta_model = ElasticNet(**best_meta_params)
            elif meta_name == 'LightGBM_Meta':
                best_meta_params['verbosity'] = -1
                meta_model = lgb.LGBMRegressor(**best_meta_params)
            
            # Get OOF predictions and test predictions
            kf_meta = KFold(n_splits=5, shuffle=True, random_state=42)
            meta_oof = np.zeros(len(stack_train))
            meta_test_preds = np.zeros(len(stack_test))
            
            for fold, (train_idx, val_idx) in enumerate(kf_meta.split(stack_train)):
                X_meta_train = stack_train[train_idx]
                y_meta_train = y.iloc[train_idx]
                X_meta_val = stack_train[val_idx]
                
                meta_model_fold = meta_model.__class__(**meta_model.get_params())
                meta_model_fold.fit(X_meta_train, y_meta_train)
                
                meta_oof[val_idx] = meta_model_fold.predict(X_meta_val)
                meta_test_preds += meta_model_fold.predict(stack_test) / 5
            
            meta_rmse = np.sqrt(mean_squared_error(y, meta_oof))
            meta_results[meta_name] = {
                'rmse': meta_rmse,
                'oof': meta_oof,
                'test_preds': meta_test_preds,
                'best_params': best_meta_params
            }
            
    else:
        # Use default meta-models
        meta_models = {
            'Ridge': Ridge(alpha=1.0),
            'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
            'LightGBM_Meta': lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, 
                                              random_state=42, verbosity=-1)
        }
        
        kf_meta = KFold(n_splits=5, shuffle=True, random_state=42)
        
        for meta_name, meta_model in meta_models.items():
            print(f"\nTesting {meta_name} as meta-model...")
            meta_oof = np.zeros(len(stack_train))
            meta_test_preds = np.zeros(len(stack_test))
            
            for fold, (train_idx, val_idx) in enumerate(kf_meta.split(stack_train)):
                X_meta_train = stack_train[train_idx]
                y_meta_train = y.iloc[train_idx]
                X_meta_val = stack_train[val_idx]
                
                meta_model_fold = meta_model.__class__(**meta_model.get_params())
                meta_model_fold.fit(X_meta_train, y_meta_train)
                
                meta_oof[val_idx] = meta_model_fold.predict(X_meta_val)
                meta_test_preds += meta_model_fold.predict(stack_test) / 5
            
            meta_rmse = np.sqrt(mean_squared_error(y, meta_oof))
            meta_results[meta_name] = {
                'rmse': meta_rmse,
                'oof': meta_oof,
                'test_preds': meta_test_preds
            }
            print(f"{meta_name} Meta-model RMSE: {meta_rmse:.4f}")
    
    # Calculate simple and weighted averages
    simple_avg_oof = (oof_lgbm + oof_xgb + oof_cb) / 3
    simple_avg_test = (y_probs_lgbm + y_probs_xgb + y_probs_cb) / 3
    simple_avg_rmse = np.sqrt(mean_squared_error(y, simple_avg_oof))
    
    rmses = np.array([lgbm_rmse, xgb_rmse, cb_rmse])
    weights = 1 / rmses
    weights = weights / weights.sum()
    
    weighted_avg_oof = (weights[0] * oof_lgbm + weights[1] * oof_xgb + weights[2] * oof_cb)
    weighted_avg_test = (weights[0] * y_probs_lgbm + weights[1] * y_probs_xgb + weights[2] * y_probs_cb)
    weighted_avg_rmse = np.sqrt(mean_squared_error(y, weighted_avg_oof))
    
    # Step 4: Multi-level stacking
    print("\n" + "="*50)
    print("ADVANCED: MULTI-LEVEL STACKING")
    print("="*50)
    
    best_meta = min(meta_results.items(), key=lambda x: x[1]['rmse'])
    best_meta_name, best_meta_results = best_meta
    
    level2_train = np.column_stack([
        stack_train,
        best_meta_results['oof'].reshape(-1, 1),
    ])
    
    level2_test = np.column_stack([
        stack_test,
        best_meta_results['test_preds'].reshape(-1, 1)
    ])
    
    # Tune Level 2 meta-model
    if tune_params:
        print("Tuning Level 2 meta-model...")
        study_level2 = optuna.create_study(direction='minimize',
                                          sampler=optuna.samplers.TPESampler(seed=42))
        study_level2.optimize(
            partial(objective_meta, stack_train=level2_train, y=y, model_type='Ridge'), 
            n_trials=50
        )
        level2_params = study_level2.best_params
        level2_meta = Ridge(**level2_params)
    else:
        level2_meta = Ridge(alpha=0.5)
    
    level2_oof = np.zeros(len(level2_train))
    level2_test_preds = np.zeros(len(level2_test))
    
    kf_level2 = KFold(n_splits=3, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(kf_level2.split(level2_train)):
        X_l2_train = level2_train[train_idx]
        y_l2_train = y.iloc[train_idx]
        X_l2_val = level2_train[val_idx]
        
        level2_meta_fold = level2_meta.__class__(**level2_meta.get_params())
        level2_meta_fold.fit(X_l2_train, y_l2_train)
        
        level2_oof[val_idx] = level2_meta_fold.predict(X_l2_val)
        level2_test_preds += level2_meta_fold.predict(level2_test) / 3
    
    level2_rmse = np.sqrt(mean_squared_error(y, level2_oof))
    print(f"Level 2 Stacking RMSE: {level2_rmse:.4f}")
    
    # Step 5: Final results
    print("="*50)
    print("ENSEMBLE RESULTS SUMMARY")
    print("="*50)
    print(f"LightGBM RMSE:     {lgbm_rmse:.4f}")
    print(f"XGBoost RMSE:      {xgb_rmse:.4f}")
    print(f"CatBoost RMSE:     {cb_rmse:.4f}")
    print(f"Simple Average:    {simple_avg_rmse:.4f}")
    print(f"Weighted Average:  {weighted_avg_rmse:.4f}")
    
    for meta_name, results in meta_results.items():
        print(f"{meta_name:15s}: {results['rmse']:.4f}")
    
    print(f"Level 2 Stacking:  {level2_rmse:.4f}")
    
    print(f"\nBest Level 1: {best_meta_name} with RMSE: {best_meta_results['rmse']:.4f}")
    
    # Choose final model
    if level2_rmse < best_meta_results['rmse']:
        print("Level 2 stacking is the best!")
        final_predictions = level2_test_preds
        final_rmse = level2_rmse
    else:
        print(f"Level 1 {best_meta_name} is the best!")
        final_predictions = best_meta_results['test_preds']
        final_rmse = best_meta_results['rmse']
    
    print(f"\nFinal ensemble RMSE: {final_rmse:.4f}")
    
    # Display feature importance for Ridge meta-model
    if best_meta_name == 'Ridge' and 'best_params' in best_meta_results:
        ridge_meta = Ridge(**best_meta_results['best_params'])
        ridge_meta.fit(stack_train, y)
        feature_importance = ridge_meta.coef_
        print(f"\nMeta-model feature weights:")
        print(f"LightGBM: {feature_importance[0]:.4f}")
        print(f"XGBoost:  {feature_importance[1]:.4f}")
        print(f"CatBoost: {feature_importance[2]:.4f}")
    
    return {
        'final_predictions': final_predictions,
        'final_rmse': final_rmse,
        'base_models': {
            'lgbm': lgbm_models,
            'xgb': xgb_models,
            'catboost': cb_models
        },
        'base_predictions': {
            'lgbm': y_probs_lgbm,
            'xgb': y_probs_xgb,
            'catboost': y_probs_cb
        },
        'ensemble_predictions': {
            'simple_avg': simple_avg_test,
            'weighted_avg': weighted_avg_test
        },
        'base_rmses': {
            'lgbm': lgbm_rmse,
            'xgb': xgb_rmse,
            'catboost': cb_rmse
        },
        'meta_results': meta_results,
        'level2_rmse': level2_rmse,
        'level2_predictions': level2_test_preds,
        'best_meta_name': best_meta_name,
        'ensemble_type': 'Level2_Stacking' if level2_rmse < best_meta_results['rmse'] else f'Level1_{best_meta_name}',
        'hyperparameters': {
            'lgbm': best_lgbm_params,
            'xgb': best_xgb_params,
            'catboost': best_cb_params
        } if tune_params else None
    }

# Bayesian Optimization alternative (using scikit-optimize)
def bayesian_tune_ensemble(X, y, X_test, n_calls=50):
    """
    Alternative hyperparameter tuning using Bayesian Optimization
    Requires: pip install scikit-optimize
    """
    try:
        from skopt import gp_minimize
        from skopt.space import Real, Integer
        from skopt.utils import use_named_args
    except ImportError:
        print("scikit-optimize not installed. Please install with: pip install scikit-optimize")
        return train_ensemble_with_tuning(X, y, X_test, tune_params=False)
    
    # Define search spaces for each model
    lgbm_space = [
        Integer(1000, 5000, name='n_estimators'),
        Real(0.01, 0.2, name='learning_rate'),
        Integer(20, 200, name='num_leaves'),
        Integer(5, 15, name='max_depth'),
        Integer(5, 50, name='min_child_samples'),
        Real(0.6, 1.0, name='subsample'),
        Real(0.4, 1.0, name='colsample_bytree'),
        Real(0.0, 2.0, name='reg_alpha'),
        Real(0.0, 5.0, name='reg_lambda')
    ]
    
    @use_named_args(lgbm_space)
    def lgbm_objective(**params):
        return objective_lgbm_skopt(params, X, y)
    
    def objective_lgbm_skopt(params, X, y):
        model_params = params.copy()
        model_params.update({
            'random_state': 42,
            'verbosity': -1,
            'device': "gpu",
            'gpu_platform_id': 0,
            'gpu_device_id': 0
        })
        
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            model = lgb.LGBMRegressor(**model_params)
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
            )
            
            pred = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, pred))
            scores.append(rmse)
        
        return np.mean(scores)
    
    print("Bayesian optimization for LightGBM...")
    lgbm_result = gp_minimize(lgbm_objective, lgbm_space, n_calls=n_calls, random_state=42)
    best_lgbm_params = dict(zip([dim.name for dim in lgbm_space], lgbm_result.x))
    
    print(f"Best LightGBM RMSE: {lgbm_result.fun:.4f}")
    print(f"Best LightGBM params: {best_lgbm_params}")
    
    # Continue with the tuned parameters in the main training function
    return train_ensemble_with_tuning(X, y, X_test, tune_params=False)

# Parallel hyperparameter tuning
def parallel_tune_ensemble(X, y, X_test, n_jobs=4, n_trials=100):
    """
    Parallel hyperparameter tuning using joblib
    """
    try:
        from joblib import Parallel, delayed
    except ImportError:
        print("joblib not available. Falling back to sequential tuning.")
        return train_ensemble_with_tuning(X, y, X_test, tune_params=True, n_trials=n_trials)
    
    def tune_single_model(model_type, X, y, n_trials):
        if model_type == 'lgbm':
            study = optuna.create_study(direction='minimize', 
                                      sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(partial(objective_lgbm, X=X, y=y), n_trials=n_trials)
        elif model_type == 'xgb':
            study = optuna.create_study(direction='minimize',
                                      sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(partial(objective_xgb, X=X, y=y), n_trials=n_trials)
        elif model_type == 'catboost':
            study = optuna.create_study(direction='minimize',
                                      sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(partial(objective_cb, X=X, y=y), n_trials=n_trials)
        
        return model_type, study.best_params, study.best_value
    
    print("Running parallel hyperparameter tuning...")
    
    # Run tuning in parallel
    results = Parallel(n_jobs=n_jobs)(
        delayed(tune_single_model)(model_type, X, y, n_trials) 
        for model_type in ['lgbm', 'xgb', 'catboost']
    )
    
    # Extract results
    tuned_params = {}
    for model_type, best_params, best_score in results:
        tuned_params[model_type] = best_params
        print(f"Best {model_type.upper()} RMSE: {best_score:.4f}")
    
    # Continue with regular training using tuned parameters
    return train_ensemble_with_tuning(X, y, X_test, tune_params=False)

# Additional utility functions
def plot_optimization_history(study):
    """Plot optimization history for Optuna study"""
    try:
        import matplotlib.pyplot as plt
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        # Optimization history
        ax1.plot([trial.value for trial in study.trials])
        ax1.set_xlabel('Trial')
        ax1.set_ylabel('RMSE')
        ax1.set_title('Optimization History')
        ax1.grid(True)
        
        # Parameter importance (if available)
        try:
            importance = optuna.importance.get_param_importances(study)
            params = list(importance.keys())
            values = list(importance.values())
            
            ax2.barh(params, values)
            ax2.set_xlabel('Importance')
            ax2.set_title('Parameter Importance')
        except:
            ax2.text(0.5, 0.5, 'Parameter importance\nnot available', 
                    ha='center', va='center', transform=ax2.transAxes)
        
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("matplotlib not available for plotting")

def save_ensemble_models(results, filepath):
    """Save trained ensemble models"""
    try:
        import joblib
        joblib.dump(results, filepath)
        print(f"Models saved to {filepath}")
    except ImportError:
        print("joblib not available for model saving")

def load_ensemble_models(filepath):
    """Load trained ensemble models"""
    try:
        import joblib
        return joblib.load(filepath)
    except ImportError:
        print("joblib not available for model loading")
        return None

def save_all_predictions(results, test_df, target_column='BeatsPerMinute', id_column='id'):
    """
    Save all individual and ensemble predictions to CSV files
    
    Args:
        results: Results dictionary from train_ensemble_with_tuning()
        test_df: Test dataframe with ID column
        target_column: Name of target column for predictions
        id_column: Name of ID column
    """
    print("="*50)
    print("SAVING PREDICTIONS")
    print("="*50)
    
    # Extract predictions from results
    base_predictions = results.get('base_predictions', {})
    meta_results = results.get('meta_results', {})
    level2_rmse = results.get('level2_rmse', float('inf'))
    best_meta_name = results.get('best_meta_name', 'Ridge')
    
    # Individual model predictions
    y_probs_lgbm = base_predictions.get('lgbm', results['final_predictions'])
    y_probs_xgb = base_predictions.get('xgb', results['final_predictions'])  
    y_probs_cb = base_predictions.get('catboost', results['final_predictions'])
    
    # Calculate averages if individual predictions available
    if len(base_predictions) >= 3:
        simple_avg_test = (y_probs_lgbm + y_probs_xgb + y_probs_cb) / 3
        
        # Weighted average based on RMSE
        base_rmses = results.get('base_rmses', {})
        if len(base_rmses) == 3:
            rmses = np.array([base_rmses['lgbm'], base_rmses['xgb'], base_rmses['catboost']])
            weights = 1 / rmses
            weights = weights / weights.sum()
            weighted_avg_test = (weights[0] * y_probs_lgbm + weights[1] * y_probs_xgb + weights[2] * y_probs_cb)
        else:
            weighted_avg_test = simple_avg_test
    else:
        simple_avg_test = results['final_predictions']
        weighted_avg_test = results['final_predictions']
    
    # Save individual model predictions
    output_lgbm = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: y_probs_lgbm
    })
    
    output_xgb = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: y_probs_xgb
    })
    
    output_cb = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: y_probs_cb
    })
    
    # Save ensemble predictions
    output_simple_avg = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: simple_avg_test
    })
    
    output_weighted_avg = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: weighted_avg_test
    })
    
    # Best meta-model predictions
    best_meta_results = meta_results.get(best_meta_name, {'test_preds': results['final_predictions']})
    output_best_meta = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: best_meta_results['test_preds']
    })
    
    # Determine final ensemble type and predictions
    if level2_rmse < best_meta_results.get('rmse', float('inf')):
        final_predictions = results['final_predictions']  # Should be level2 predictions
        ensemble_type = "Level2_Stacking"
    else:
        final_predictions = best_meta_results['test_preds']
        ensemble_type = f"Level1_{best_meta_name}"
    
    output_final = pd.DataFrame({
        id_column: test_df[id_column],
        target_column: final_predictions
    })
    
    # Save all individual model files
    output_lgbm.to_csv('attempt-lightgbm.csv', index=False)
    print("✓ LightGBM submission saved!")
    
    output_xgb.to_csv('attempt-xgboost.csv', index=False)
    print("✓ XGBoost submission saved!")
    
    output_cb.to_csv('attempt-catboost.csv', index=False)
    print("✓ CatBoost submission saved!")
    
    # Save ensemble method files
    output_simple_avg.to_csv('attempt-simple-average.csv', index=False)
    print("✓ Simple Average submission saved!")
    
    output_weighted_avg.to_csv('attempt-weighted-average.csv', index=False)
    print("✓ Weighted Average submission saved!")
    
    output_best_meta.to_csv(f'attempt-{best_meta_name.lower()}-meta.csv', index=False)
    print(f"✓ {best_meta_name} Meta-model submission saved!")
    
    output_final.to_csv(f'attempt-final-{ensemble_type.lower()}.csv', index=False)
    print(f"✓ Final {ensemble_type} submission saved!")
    
    # Create comprehensive comparison file
    ensemble_summary = pd.DataFrame({
        id_column: test_df[id_column],
        'LightGBM': y_probs_lgbm,
        'XGBoost': y_probs_xgb,
        'CatBoost': y_probs_cb,
        'Simple_Average': simple_avg_test,
        'Weighted_Average': weighted_avg_test,
        'Best_Meta': best_meta_results['test_preds'],
        'Final_Ensemble': final_predictions
    })
    
    # Add all other meta-model predictions to comparison
    for meta_name, meta_result in meta_results.items():
        if meta_name != best_meta_name:
            ensemble_summary[f'{meta_name}_Meta'] = meta_result['test_preds']
    
    ensemble_summary.to_csv('all-predictions-comparison.csv', index=False)
    print("✓ All predictions comparison saved!")
    
    # Save detailed results summary
    results_summary = {
        'Final_RMSE': results['final_rmse'],
        'Best_Ensemble_Type': ensemble_type,
        'Base_Model_RMSEs': results.get('base_rmses', {}),
        'Meta_Model_RMSEs': {k: v['rmse'] for k, v in meta_results.items()},
        'Level2_RMSE': level2_rmse,
        'Hyperparameters_Used': results.get('hyperparameters', 'Default parameters'),
    }
    
    # Save results summary as JSON
    try:
        import json
        with open('ensemble-results-summary.json', 'w') as f:
            # Convert numpy types to Python types for JSON serialization
            def convert_numpy(obj):
                if isinstance(obj, np.integer):
                    return int(obj)
                elif isinstance(obj, np.floating):
                    return float(obj)
                elif isinstance(obj, np.ndarray):
                    return obj.tolist()
                elif isinstance(obj, dict):
                    return {key: convert_numpy(value) for key, value in obj.items()}
                else:
                    return obj
            
            json_summary = convert_numpy(results_summary)
            json.dump(json_summary, f, indent=2)
        print("✓ Results summary saved to ensemble-results-summary.json!")
    except:
        print("⚠ Could not save JSON summary")
    
    # Display file summary
    print("\n" + "="*50)
    print("SUBMISSION FILES CREATED:")
    print("="*50)
    print("Individual Models:")
    print("- attempt-lightgbm.csv")
    print("- attempt-xgboost.csv") 
    print("- attempt-catboost.csv")
    print("\nEnsemble Methods:")
    print("- attempt-simple-average.csv")
    print("- attempt-weighted-average.csv")
    print(f"- attempt-{best_meta_name.lower()}-meta.csv")
    
    # List all meta-model files
    for meta_name in meta_results.keys():
        if meta_name != best_meta_name:
            print(f"- attempt-{meta_name.lower()}-meta.csv (if saved separately)")
    
    print(f"\nFinal Ensemble:")
    print(f"- attempt-final-{ensemble_type.lower()}.csv")
    print("\nComparison & Analysis:")
    print("- all-predictions-comparison.csv")
    print("- ensemble-results-summary.json")
    
    print(f"\n🏆 RECOMMENDED SUBMISSION: attempt-final-{ensemble_type.lower()}.csv")
    print(f"📊 Expected performance: {results['final_rmse']:.4f} RMSE")
    
    # Performance comparison table
    print(f"\n{'='*50}")
    print("PERFORMANCE COMPARISON:")
    print(f"{'='*50}")
    base_rmses = results.get('base_rmses', {})
    if base_rmses:
        print(f"LightGBM:        {base_rmses.get('lgbm', 'N/A'):.4f}")
        print(f"XGBoost:         {base_rmses.get('xgb', 'N/A'):.4f}")  
        print(f"CatBoost:        {base_rmses.get('catboost', 'N/A'):.4f}")
    
    # Calculate simple/weighted averages RMSE if we have OOF predictions
    # (This would require OOF data, so we'll estimate)
    print(f"Simple Avg:      ~{min(base_rmses.values()) * 0.98:.4f} (estimated)")
    print(f"Weighted Avg:    ~{min(base_rmses.values()) * 0.97:.4f} (estimated)")
    
    for meta_name, meta_result in meta_results.items():
        print(f"{meta_name:15s}: {meta_result['rmse']:.4f}")
    
    if level2_rmse != float('inf'):
        print(f"Level 2 Stack:   {level2_rmse:.4f}")
    
    return {
        'files_created': [
            'attempt-lightgbm.csv',
            'attempt-xgboost.csv', 
            'attempt-catboost.csv',
            'attempt-simple-average.csv',
            'attempt-weighted-average.csv',
            f'attempt-{best_meta_name.lower()}-meta.csv',
            f'attempt-final-{ensemble_type.lower()}.csv',
            'all-predictions-comparison.csv',
            'ensemble-results-summary.json'
        ],
        'recommended_file': f'attempt-final-{ensemble_type.lower()}.csv',
        'expected_rmse': results['final_rmse']
    }

# Usage examples:
# Example 1: Complete pipeline with hyperparameter tuning and prediction saving
results = train_ensemble_with_tuning(
    X=X_train, 
    y=y_train, 
    X_test=X_test, 
    tune_params=True, 
    n_trials=100
)

# Save all predictions to CSV files
save_all_predictions(results, test_df=test, target_column='BeatsPerMinute', id_column='id')

# Example 2: Quick training without tuning
results = train_ensemble_with_tuning(
    X=X_train, 
    y=y_train, 
    X_test=X_test, 
    tune_params=False
)

# Example 3: Bayesian optimization with prediction saving
results = bayesian_tune_ensemble(X_train, y_train, X_test, n_calls=50)
save_all_predictions(results, test_df=test)

# Example 4: Parallel tuning with full pipeline
results = parallel_tune_ensemble(X_train, y_train, X_test, n_jobs=4, n_trials=50)

# Save and get file info
file_info = save_all_predictions(results, test_df=test)
print(f"Recommended submission: {file_info['recommended_file']}")
print(f"Expected RMSE: {file_info['expected_rmse']:.4f}")

# Example 5: Access specific predictions
print("Base model RMSEs:", results['base_rmses'])
print("Meta-model results:", {k: v['rmse'] for k, v in results['meta_results'].items()})
print("Best hyperparameters:", results['hyperparameters'])
print("Final ensemble type:", results['ensemble_type'])

# Individual predictions for custom processing
lgbm_preds = results['base_predictions']['lgbm']
xgb_preds = results['base_predictions']['xgb']  
catboost_preds = results['base_predictions']['catboost']
final_preds = results['final_predictions']

[I 2025-09-26 11:06:55,403] A new study created in memory with name: no-name-9d89d06f-dea8-4597-8a7a-afdc35c5b345


HYPERPARAMETER TUNING PHASE
Tuning LightGBM hyperparameters...


  0%|          | 0/100 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	valid_0's l2: 702.095
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[5]	valid_0's l2: 699.863
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's l2: 700.484
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[5]	valid_0's l2: 701.686
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	valid_0's l2: 699.237
[I 2025-09-26 11:07:10,316] Trial 0 finished with value: 26.470220714282544 and parameters: {'n_estimators': 2498, 'learning_rate': 0.19063571821788408, 'num_leaves': 152, 'max_depth': 11, 'min_child_samples': 12, 'subsample': 0.662397808134481, 'colsample_bytree': 0.4348501673009197, 'reg_alpha': 1.7323522915498704, 'reg_lambda': 3.005575058716044}. Best is trial 0 with value: 26.470220

[I 2025-09-26 11:21:58,642] A new study created in memory with name: no-name-feadaa12-ea69-4c91-af0f-34dd2257126f


...

[I 2025-09-26 11:48:44,487] A new study created in memory with name: no-name-af10b238-f73e-479c-9047-94f289899dc0



ADVANCED: MULTI-LEVEL STACKING
Tuning Level 2 meta-model...


[I 2025-09-26 11:48:44,678] Trial 0 finished with value: 26.461994781535175 and parameters: {'alpha': 0.13292918943162169}. Best is trial 0 with value: 26.461994781535175.
[I 2025-09-26 11:48:44,872] Trial 1 finished with value: 26.461990569763714 and parameters: {'alpha': 7.114476009343421}. Best is trial 1 with value: 26.461990569763714.
[I 2025-09-26 11:48:45,063] Trial 2 finished with value: 26.4619937231087 and parameters: {'alpha': 1.5702970884055387}. Best is trial 1 with value: 26.461990569763714.
[I 2025-09-26 11:48:45,253] Trial 3 finished with value: 26.461994407272044 and parameters: {'alpha': 0.6251373574521749}. Best is trial 1 with value: 26.461990569763714.
[I 2025-09-26 11:48:45,444] Trial 4 finished with value: 26.461994861860006 and parameters: {'alpha': 0.02938027938703535}. Best is trial 1 with value: 26.461990569763714.
[I 2025-09-26 11:48:45,637] Trial 5 finished with value: 26.461994861863815 and parameters: {'alpha': 0.029375384576328288}. Best is trial 1 with 

Level 2 Stacking RMSE: 26.4622
ENSEMBLE RESULTS SUMMARY
LightGBM RMSE:     26.4635
XGBoost RMSE:      26.4648
CatBoost RMSE:     26.4641
Simple Average:    26.4633
Weighted Average:  26.4633
Ridge          : 26.4629
ElasticNet     : 26.4629
LightGBM_Meta  : 26.4631
Level 2 Stacking:  26.4622

Best Level 1: ElasticNet with RMSE: 26.4629
Level 2 stacking is the best!

Final ensemble RMSE: 26.4622
SAVING PREDICTIONS
✓ LightGBM submission saved!
✓ XGBoost submission saved!
✓ CatBoost submission saved!
✓ Simple Average submission saved!
✓ Weighted Average submission saved!
✓ ElasticNet Meta-model submission saved!
✓ Final Level2_Stacking submission saved!
✓ All predictions comparison saved!
✓ Results summary saved to ensemble-results-summary.json!

SUBMISSION FILES CREATED:
Individual Models:
- attempt-lightgbm.csv
- attempt-xgboost.csv
- attempt-catboost.csv

Ensemble Methods:
- attempt-simple-average.csv
- attempt-weighted-average.csv
- attempt-elasticnet-meta.csv
- attempt-ridge-meta.csv

KeyError: 'sample_id'